# 认识消息
大模型没有记忆，只能根据当前消息(上下文)进行响应。
很多场景下，我们需要根据历史消息(上下文)进行响应。
如果应用需要记住历史消息(上下文)，则需要再程序中维护消息列表

在langchian中，Message(消息)是模型交互的最基本单元。
代表输入或输出（input/output）

## 消息的内部结构
Role: 角色（user/assistant/system）
- 系统消息：系统提示词，用于对话开始前为模型设定角色、行为准则和上下文背景它像是给AI助手的一份工作说明书，决定了其回答问题的风格、领域和专业范围。
{"role": "system", "content": "你是个精通编程的软件架构师"}
- 用户信息：用户提示词，用于向模型发送问题或指令。
{"role": "user", "content": "你好"}
- 助手信息：模型的响应，包含问题的答案或指令的执行。
{"role": "assistant", "content": "你好，有什么我可以帮助你的吗？"}
- 工具调用的信息：模型调用外部工具或函数，用于执行更复杂的任务。
{"role": "tool", "content": "今天天气很好", "tool_call_id":
"call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"}


问题：为什么使用不同的消息类型？

明确角色 ：清晰区分系统提示、用户输入和 AI 回复

控制行为 ：通过 SystemMessage 精确控制 AI 的行为

对话历史 ：构建完整的多轮对话上下文

调试友好 ：更容易追踪和调试对话流程


Content: 消息内容

Metadata: （可选的）元数据
消息的元数据可以包含消息的发送时间、发送者、接收者等信息。

## 格式
- json

- 对象格式

举例1：JSON格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
import os

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(
    model_name="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE
)

# 初始化格式
messages = [
    {"role": "system", "content": "你是个精通编程的软件架构师"},
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好，有什么我可以帮助你的吗？"},
    {"role": "user", "content": "你好"},
]

response = model.invoke(messages)
print(response.content)


举例2：对象格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage 

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

model = init_chat_model(
    model_name="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_API_BASE
)

messages1= [
    SystemMessage(content="你是个精通编程的软件架构师"),
    HumanMessage(content="你好"),
    AIMessage(content="你好，有什么我可以帮助你的吗？"),
    HumanMessage(content="你好"),
]


# 或者
messages = [
    SystemMessage("你是个精通编程的软件架构师"),
    HumanMessage("你好"),
    AIMessage("你好，有什么我可以帮助你的吗？"),
    HumanMessage("你好"),
]

response = model.invoke(messages)
print(response.content)


## 消息对象字段说明
content ：消息内容，字段名可以省略

SystemMessage参数列表相当于SystemMessage(content="你是个精通编程的软件架构师")

HumanMessage参数列表相当于HumanMessage(content="你好")

注意这里可以带元数据字段metadata

HumanMessage(

    content="Hello!",

    name="alice", # 可选，用户名

    id="msg_123", # 可选，message的ID

    metadata={"user_id": "123456"}, # 可选，自定义元数据

)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek:deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

messages = [
    SystemMessage("你是一个信息提取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出格式：{\"speakers\": [{\"name\":\"...\", \"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)

模型加载了